# ComplaintIQ - unsupervised EDA (`03_eda_unsupervised`)

EDA for the **unsupervised** side: clustering complaint themes, topic modeling over
narratives, and retrieving similar historical complaints. No label to train against, so
this notebook characterizes *structure*. **EDA only - no modeling.** (Target-centric EDA
lives in `02_eda_supervised.ipynb`.)

Shares the same **data foundation** as `02` (load + schema -> inspect -> data
quality -> missingness), run on a fixed-seed **sample** because distance-based methods
cannot scan 16.5M rows, then diverges into structure-focused sections.

## EDA objectives
We explore structure to make defensible choices for the unsupervised tasks, since there
is no label to optimize against. Each section produces evidence for a decision:

- **Trust the data / scope the sample.** Confirm the schema and draw a reproducible
  sample distance methods can handle.
- **Choose the text representation.** Size the vocabulary and term-frequency skew for
  TF-IDF `min_df` / `max_df` / `max_features` (or embeddings).
- **Prepare the feature space.** Find scale differences and redundancy a distance metric
  would distort, so we know what to standardize / drop.
- **Set an evaluation yardstick.** Read `product x issue` co-occurrence as label-free
  themes a good clustering should recover.
- **Clean for structure.** Quantify near-duplicate narratives to dedup before
  clustering / retrieval.

> **Note:** Spark reads the Parquet and draws the fixed-seed, target-stratified sample;
> the rest of the notebook works on that sample in pandas (`.toPandas()`).

> **Go deeper:**
> - [CFPB Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/): *the source dataset, its fields, and how complaints are collected - start here if the columns are unfamiliar.*
> - [scikit-learn: clustering overview](https://scikit-learn.org/stable/modules/clustering.html): *k-means vs hierarchical vs density methods and when each fits, ~12 min.*

## How to read this notebook
The analysis is written to be **reproducible** and followed by someone new to the data:
columns are loaded by explicit name, every sample uses the fixed `RANDOM_STATE`, and
each figure names the decision it drives.

> **Note:** a fact to pin down.
>
> **Tip:** a good habit.
>
> **Warning:** something that would bias a downstream model if ignored.
>
> **Go deeper:** a curated link for background (what it is, why click).

Each figure carries a **What you're seeing / Notice / Why it matters** caption, where
*Why it matters* states the concrete modeling or parameter decision the evidence drives.

## Setup

Same imports and house theme as the other notebooks, plus the shared token helper.

In [ ]:
import sys
from pathlib import Path

# Import shared helpers from the local complaintiq package. Walk up from the cwd
# to find the repo's src/ dir, so this works whether the notebook lives in
# notebooks/ or notebooks/appendix/, locally or in a Databricks Git folder.
_here = Path.cwd()
_src = None
for _p in [_here, *_here.parents]:
    if (_p / "src" / "complaintiq").exists():
        _src = str(_p / "src")
        break
if _src and _src not in sys.path:
    sys.path.insert(0, _src)

In [ ]:
from __future__ import annotations
from collections.abc import Iterable
from complaintiq import RANDOM_STATE, np, pd, plt, sns, print_versions  # shared setup
from complaintiq.text import STOPWORDS, tokenize, top_document_terms, top_bigrams

# The tools we lean on across the project.
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# House plotting theme: clean grid + colorblind-safe palette, used everywhere.
sns.set_theme(style="whitegrid", palette="colorblind", context="notebook", font_scale=1.1)
plt.rcParams["figure.figsize"] = (10, 6)

# The one true seed, so any sample below is reproducible run-to-run.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("python :", sys.version.split()[0])
print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("seaborn:", sns.__version__)

In [ ]:
import re
from collections import Counter

# A tiny stoplist keeps the token views readable without pulling in a modeling
# library. This is descriptive counting (EDA), not feature extraction.

---
## 1. Load a representative sample + schema check *(shared foundation)*

Clustering / pairwise similarity cannot run over 16.5M documents directly, so EDA
establishes a reproducible **sample** to prototype on. Spark reads the narrative-only
Parquet (dense text), then draws a fixed-seed sample **stratified on `monetary_relief`**
so the rare positive class is represented; only that sample is pulled into pandas.

> **Tip (Spark):** filter and sample in Spark first, then `.toPandas()` the subset, never
> pull the full corpus into the driver.

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F

# Resolve the data directory: the UC volume when running on Databricks,
# else the local ../data produced by `make parquet`.
VOLUME_DIR = Path("/Volumes/workspace/complaintiq/data")
data_dir = VOLUME_DIR if VOLUME_DIR.exists() else Path("..") / "data"
nar_path = data_dir / "complaints_narrative_only.parquet"
full_path = data_dir / "complaints.parquet"

use_cols = [
    "complaint_id",
    "complaint_text",
    "complaint_text_length",
    "product",
    "issue",
    "sub_product",
    "state",
    "zip_code",
    "company",
    "submitted_via",
    "timely_response",
    "monetary_relief",
    "company_response_to_consumer",
]

src = nar_path if nar_path.exists() else full_path
if not src.exists():
    raise FileNotFoundError("No Parquet in data/. Run `make parquet NARRATIVE_ONLY=1` first.")

src_sdf = spark.read.parquet(str(src))
present = set(src_sdf.columns)
need = set(use_cols) | ({"has_narrative"} if src == full_path else set())
absent = need - present
assert not absent, f"Parquet missing expected columns {absent}. Regenerate with `make parquet`."

read_cols = use_cols + (["has_narrative"] if src == full_path else [])
docs = src_sdf.select(*read_cols)
if src == full_path:
    docs = docs.filter(F.col("has_narrative")).drop("has_narrative")

docs = docs.dropna(subset=["complaint_text"])
# Distance-based methods can't scan the full corpus; draw a fixed-seed sample in
# Spark and pull only that subset into pandas for the rest of the notebook.
total = docs.count()
SAMPLE_N = min(total, 25000)
# Stratify on the target: monetary relief is rare, so a uniform sample can miss
# it. Draw an exact per-class quota via a groupby + window: count each class,
# take its proportional share, then keep that many rows ranked in random order
# within the class. Proportional (not equal) keeps the true positive rate the
# EDA reports, while guaranteeing the rare class is represented.
from pyspark.sql import Window

frac = min(1.0, SAMPLE_N / total) if total else 0.0
class_counts = {
    r["monetary_relief"]: r["count"] for r in docs.groupBy("monetary_relief").count().collect()
}
per_class = {c: int(round(n * frac)) for c, n in class_counts.items()}
_rn = F.row_number().over(Window.partitionBy("monetary_relief").orderBy(F.rand(RANDOM_STATE)))
_cond = F.lit(False)
for _c, _k in per_class.items():
    _cond = _cond | ((F.col("monetary_relief") == F.lit(_c)) & (F.col("_rn") <= F.lit(_k)))
sample = docs.withColumn("_rn", _rn).filter(_cond).drop("_rn").toPandas().reset_index(drop=True)
print(f"per-class sample sizes: {per_class}")
print(f"Corpus available: {total:,} narratives; working sample: {len(sample):,}")

---
## 2. Inspect: first pass *(shared foundation)*

The same look-first pass as `02`, on the sampled pandas frame.

> **Go deeper:**
> - [PySpark: DataFrame.describe](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.describe.html): *the per-column numeric summary, for when you profile in Spark before sampling, ~3 min.*

In [ ]:
print("First 5 rows:")
display(sample.head())
print("\nData types:")
print(sample.dtypes)
print("\nNumeric summary:")
display(sample.describe())
print("\nStructure:")
sample.info()

---
## 3. Data quality *(shared foundation)*

The same integrity checks as `02`, on the sample.

In [ ]:
# Data quality checks that matter to any downstream task (supervised or unsupervised).
print("rows:", f"{len(sample):,}")
print("duplicate complaint_id values:", int(sample["complaint_id"].duplicated().sum()))
print("fully duplicated rows:        ", int(sample.duplicated().sum()))

# A zip_code should map to a single state; count zips seen with more than one.
zip_state_counts = (
    sample.dropna(subset=["zip_code", "state"])
    .groupby("zip_code", observed=True)["state"]
    .nunique()
)
print(
    f"\nzip_codes tied to >1 state: {int((zip_state_counts > 1).sum()):,} of {len(zip_state_counts):,} distinct zips"
)

# High-cardinality tail: how concentrated / how many singletons?
company_counts = sample["company"].value_counts()
print(f"\ncompanies: {sample['company'].nunique():,} distinct")
print(f"  top company = {company_counts.iloc[0] / len(sample):.2%} of rows")
print(f"  companies appearing exactly once: {int((company_counts == 1).sum()):,}")

> **What you're seeing:** integrity checks looking for duplicate keys/rows, whether a zip maps
> to one state, and how concentrated the high-cardinality `company` field is.
>
> **Notice:** watch for duplicate rows, zip/state inconsistencies, and a long tail of
> companies seen only once.
>
> **Why it matters:** duplicates must be resolved before splitting or clustering (they
> bias both); zip/state inconsistency decides whether geo is usable; the cardinality
> tail decides how you encode or group rare categories.

---
## 4. Missingness *(shared foundation)*

Null fraction per column in the sample (pandas, since the sample is already local).

> **Go deeper:**
> - [PySpark: handling missing data (DataFrame.na)](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.na.html): *how Spark represents and handles nulls when you profile the full corpus, ~8 min.*

In [ ]:
missing = sample.isna().mean().sort_values(ascending=False)
print("Missing fraction by column (sample):")
print(missing.round(4))
ax = sns.barplot(
    x=missing.values, y=missing.index, hue=missing.index, palette="colorblind", legend=False
)
ax.set_xlabel("fraction missing")
ax.set_ylabel("")
ax.set_title("Missingness by column (sample)")
plt.tight_layout()
plt.show()

---
## 5. Label context: for interpretation only *(cross-cutting)*

We do not train on a label, but the outcome mix helps *interpret* clusters later.

> **Warning (leakage for retrieval/clustering):** `monetary_relief` and
> `company_response_to_consumer` are **post-resolution**. Use them to *interpret*
> structure, never as clustering or similarity features, that leaks the outcome into
> the structure you claim to discover.

In [ ]:
print("monetary_relief in the sample:")
print(sample["monetary_relief"].value_counts().sort_index())
print(f"positive rate (sample): {sample['monetary_relief'].mean():.4%}")
print("\ncompany_response_to_consumer distribution (sample):")
display(sample["company_response_to_consumer"].value_counts(normalize=True).round(4))

> **What you're seeing:** the outcome distribution among sampled narratives.
>
> **Notice:** the label is available and imbalanced, as in `02`.
>
> **Why it matters:** it is a free yardstick for interpreting clusters, but a forbidden
> *input*, per the warning above.

---
## 6. Corpus characterization *(unsupervised)*

Topic modeling and retrieval depend on corpus shape: document length, vocabulary size,
and how skewed term frequencies are.

> **Go deeper:**
> - [scikit-learn: TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html): *the min_df / max_df / max_features knobs this EDA is sizing, ~8 min.*
> - [Zipf's law](https://en.wikipedia.org/wiki/Zipf%27s_law): *why a few terms dominate and a long tail occurs once, ~4 min.*

In [ ]:
sample["n_tokens"] = sample["complaint_text"].map(lambda t: len(tokenize(t)))
print("tokens per narrative (sample):")
display(sample["n_tokens"].describe())
ax = sns.histplot(sample["n_tokens"], bins=50, log_scale=(True, False))
ax.set_xlabel("tokens per narrative (log scale)")
ax.set_ylabel("narratives")
ax.set_title("Narrative length in tokens (sample)")
plt.tight_layout()
plt.show()

In [ ]:
vocab = Counter()
for doc in sample["complaint_text"]:
    vocab.update(tokenize(doc))
total_tokens = sum(vocab.values())
hapax = sum(1 for _, n in vocab.items() if n == 1)
print(f"total tokens (sample): {total_tokens:,}")
print(f"vocabulary size:       {len(vocab):,}")
print(f"hapax legomena (freq 1): {hapax:,} ({hapax / max(len(vocab), 1):.1%} of vocab)")
print("\ntop bigrams (sample):", [b for b, _ in top_bigrams(sample["complaint_text"], k=12)])

In [ ]:
top = vocab.most_common(20)
terms = [t for t, _ in top]
freqs = [n for _, n in top]
ax = sns.barplot(x=freqs, y=terms, hue=terms, palette="colorblind", legend=False)
ax.set_xlabel("term frequency (sample)")
ax.set_ylabel("")
ax.set_title("Top 20 corpus terms")
plt.tight_layout()
plt.show()

> **What you're seeing:** token-length distribution, vocabulary size + hapax
> fraction, top terms and bigrams.
>
> **Notice:** a large vocabulary with a big hapax tail and a few dominant terms.
>
> **Why it matters:** fixes TF-IDF `min_df` (drop the hapax tail), `max_df` /
> `max_features` (tame dominant terms), and confirms a high-dimensional sparse space.

---
## 7. Feature-space structure for distance-based methods *(unsupervised)*

Clustering and nearest-neighbor retrieval measure *distance*, so scale and redundancy
matter: a large-range feature dominates, and correlated features double-count.

> **Go deeper:**
> - [scikit-learn: StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html): *standardizing features so no single scale dominates a distance metric, ~4 min.*
> - [Curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality): *why distances get less meaningful in high-dimensional sparse spaces, ~6 min.*

In [ ]:
num = pd.DataFrame(
    {
        "complaint_text_length": sample["complaint_text_length"].astype(float),
        "n_tokens": sample["n_tokens"].astype(float),
        "timely_response": sample["timely_response"].astype(int),
    }
)
print("Numeric feature scales (note the very different ranges):")
display(num.describe())
ax = sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
ax.set_title("Correlation among numeric features")
plt.tight_layout()
plt.show()

> **What you're seeing:** numeric feature ranges and their correlation.
>
> **Notice:** ranges differ by orders of magnitude; length and n_tokens are near-collinear.
>
> **Why it matters:** mandates standardization before any distance metric, and dropping
> one of the collinear length features so it does not double-count.

---
## 8. Category co-occurrence *(unsupervised)*

`product x issue` co-occurrence shows the dominant combinations a good clustering should roughly recover.

> **What "product x issue" means:** each complaint has two intake fields, `product` (what it is
> about, e.g. "Credit card") and `issue` (the problem, e.g. "Fee problem"). Their cross-tabulation
> (`pd.crosstab` below) is a grid counting how many complaints fall in each (product, issue) pair.
> A few pairs dominate: those are real, coherent themes.
>
> **Why it is a "yardstick":** clustering has no label to grade against, so we need something to
> compare the clusters to. These product x issue blocks are themes that already exist in the data,
> defined by fields the clustering never sees (it reads only the narrative text). A good clustering
> should roughly rediscover them, so agreement with this grid (ARI / NMI in `05`) measures cluster
> quality. It is a measuring stick only, never a feature - using it as input would be circular.

In [ ]:
top_products = sample["product"].value_counts().head(6).index
top_issues = sample["issue"].value_counts().head(8).index
top_subset = sample[sample["product"].isin(top_products) & sample["issue"].isin(top_issues)]
product_issue_crosstab = pd.crosstab(top_subset["product"], top_subset["issue"])
ax = sns.heatmap(product_issue_crosstab, annot=True, fmt="d", cmap="viridis")
ax.set_title("product x issue co-occurrence (sample, top values)")
ax.set_xlabel("issue")
ax.set_ylabel("product")
plt.tight_layout()
plt.show()

> **What you're seeing:** how often each product/issue pair co-occurs.
>
> **Notice:** a few blocks dominate; most pairs rarely occur.
>
> **Why it matters:** these blocks are a label-free yardstick for judging whether
> discovered clusters correspond to real themes.

---
## 9. Near-duplicate & template narratives *(unsupervised)*

CFPB narratives include many templated submissions. Duplicates form dense artificial
clusters and dominate similarity search, so quantify them.

> **Go deeper:**
> - [Locality-sensitive hashing](https://en.wikipedia.org/wiki/Locality-sensitive_hashing): *detecting near-duplicates beyond exact matches at scale, ~8 min.*
> - [datasketch (MinHash LSH)](https://ekzhu.com/datasketch/): *a practical library for near-duplicate detection on large corpora, optional.*

In [ ]:
text_value_counts = sample["complaint_text"].value_counts()
dup_texts = int((text_value_counts > 1).sum())
dup_rows = int(text_value_counts[text_value_counts > 1].sum())
print(f"distinct narratives repeated verbatim: {dup_texts:,}")
print(
    f"rows involved in an exact duplicate:   {dup_rows:,} ({dup_rows / len(sample):.2%} of sample)"
)
print(f"most-repeated narrative appears:        {int(text_value_counts.max())} times")
print("\nMost-repeated narratives (truncated):")
for text, n in text_value_counts.head(5).items():
    print(f"  x{n:>4}  {' '.join(str(text).split())[:100]!r}")

> **What you're seeing:** how many narratives repeat verbatim and how concentrated.
>
> **Notice:** a nontrivial share are exact duplicates/templates.
>
> **Why it matters:** decides a dedup / down-weight step before clustering & retrieval;
> exact-match counts here set the floor, and near-duplicate detection (LSH) is the
> follow-up on the full corpus.

---
## 10. Takeaways: unsupervised decisions

> **Decisions this EDA supports:**
> - **Foundation mirrors `02`:** schema, first-pass inspect, data quality, and
>   missingness all run here too, on a reproducible sample.
> - **Sample first** - 16.5M narratives are too many for distance methods.
> - **Text representation:** high-dimensional, sparse, long hapax tail: tune
>   `min_df` / `max_df` / `max_features` (or move to embeddings).
> - **Feature prep:** standardize numeric features; drop one of the collinear lengths.
> - **Evaluation yardstick:** `product x issue` blocks are label-free themes.
> - **Cleaning:** dedup / down-weight templated narratives first.
> - **Leakage:** the label is context for interpretation, never a feature (section 5).